# LLaMA MLP（SwiGLU）手撕实现

## SwiGLU
$$\text{MLP}(x)=W_2\big(\text{SiLU}(W_1 x)\odot W_3 x\big),\quad \text{SiLU}(x)=x\sigma(x)$$
- 三个投影 `gate_proj=W1 / up_proj=W3 / down_proj=W2`，**均无 bias**。
- 门控：`SiLU(W1 x)` 是门，`W3 x` 是信息，逐元素相乘再 `W2` 投影回原维。
- **hidden_dim 的 $2/3$ 缩放**：GLU 系列有 3 个矩阵，参数量约为标准 4d MLP 的 $3/2$ 倍，故把 hidden_dim 乘 $2/3$ 使总参数与 $4d$ MLP 持平，再向上取整到 `multiple_of`（如 256）对齐硬件。
- 相比 ReLU/GELU MLP，SwiGLU 平滑门控，效果更好，是 LLaMA 系列标配。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LlamaMLP(nn.Module):
    def __init__(self, dim, hidden_dim=None, multiple_of=256):
        super().__init__()
        if hidden_dim is None:
            hidden_dim = 4 * dim
        hidden_dim = int(2 * hidden_dim / 3)                       # SwiGLU 的 2/3 缩放
        hidden_dim = multiple_of * ((hidden_dim + multiple_of - 1) // multiple_of)  # 向上取整到 multiple_of
        self.gate_proj = nn.Linear(dim, hidden_dim, bias=False)     # W1
        self.up_proj   = nn.Linear(dim, hidden_dim, bias=False)     # W3
        self.down_proj = nn.Linear(hidden_dim, dim, bias=False)     # W2

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

In [ ]:
# 验证：shape + 参数量
torch.manual_seed(0)
dim = 512
mlp = LlamaMLP(dim)
x = torch.randn(2, 10, dim)
print('output shape:', mlp(x).shape)

n_params = sum(p.numel() for p in mlp.parameters())
# hidden_dim 计算
h = int(2 * 4 * dim / 3)
h = 256 * ((h + 255) // 256)
print(f'hidden_dim={h}, 三矩阵参数量≈{3*dim*h}, 实际={n_params}')
print('与标准 4d MLP 参数量 8d^2=', 8*dim*dim, '接近:', abs(n_params - 8*dim*dim) / (8*dim*dim) < 0.1)

## 小结 / 易错点
- 三个 Linear 都 `bias=False`，LLaMA 全程几乎不用 bias（Norm 的 weight 除外）。
- `SiLU(W1 x) * W3 x` 顺序：门在前、信息在后，等价但写法要一致。
- $2/3$ 缩放是为对齐标准 MLP 参数量，不是超参魔法；`multiple_of` 为 GPU 对齐。
- 推理时三个小 GEMM 可融合/重排，训练时一般保持三个独立 Linear。